## 01. 횡단보도 중구 필터 + 좌표 추출

In [9]:
import sys
sys.path.insert(0, '..')
from utils import *

import pandas as pd
import geopandas as gpd
import re

# === 횡단보도 로드 ===
cw = pd.read_csv(CROSSWALK, encoding='cp949')

# 중구만
cw_junggu = cw[cw['시군구명'] == '중구'].copy()
print(f"전체: {len(cw)} → 중구: {len(cw_junggu)}")
print(f"NODE: {(cw_junggu['노드링크 유형']=='NODE').sum()}, LINK: {(cw_junggu['노드링크 유형']=='LINK').sum()}")

# NODE만 추출 (횡단보도 = 점 위치)
cw_node = cw_junggu[cw_junggu['노드링크 유형'] == 'NODE'].copy()

# WKT에서 좌표 추출
def parse_point(wkt):
    if pd.isna(wkt):
        return None, None
    m = re.match(r'POINT\(([\d.]+)\s+([\d.]+)\)', wkt)
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None

cw_node[['lon', 'lat']] = cw_node['노드 WKT'].apply(
    lambda x: pd.Series(parse_point(x))
)

# 좌표 못 뽑은 거 확인
print(f"좌표 추출 성공: {cw_node['lon'].notna().sum()} / {len(cw_node)}")
print(cw_node[['노드 ID', 'lon', 'lat', '읍면동명']].head(10))

전체: 31080 → 중구: 1098
NODE: 674, LINK: 424
좌표 추출 성공: 674 / 674
      노드 ID         lon        lat   읍면동명
1082  10350  126.992528  37.568267    입정동
1085  10352  126.995313  37.568491    산림동
1086  10353  126.996107  37.568559    산림동
1087  10357  127.001861  37.569543    방산동
1088  10358  127.000254  37.569127    주교동
1089  10359  127.002325  37.569435    방산동
1090   8452  127.009042  37.556655    신당동
1091   8459  127.007638  37.549695    신당동
1092   8469  127.009682  37.553173    신당동
1093   9774  127.002234  37.552969  장충동2가


## 02. 사고 데이터 필터 (65세↑ + 횡단중)

In [10]:
# === 사고 데이터 로드 + 필터 ===
acc = pd.read_csv(ACCIDENTS)

elderly_cross = acc[
    (acc['dmge_age_2'] == '65세 이상') &
    (acc['acdnt_dc'].str.contains('횡단', na=False))
].copy()

print(f"65세↑ + 횡단중: {len(elderly_cross)}건")
print(elderly_cross[['acdnt_no', 'lon', 'lat']].head())

65세↑ + 횡단중: 342건
            acdnt_no         lon        lat
15  2010011700100467  126.977640  37.561026
55  2010022400100249  127.016166  37.565319
76  2010031200100224  127.003907  37.564700
77  2010031200100313  126.996165  37.564244
89  2010040200100367  126.981630  37.562018


## 03. T1 생성 (사고 ↔ 횡단보도 매칭)

In [ ]:
from shapely.geometry import Point

# GeoDataFrame 변환
gdf_acc = gpd.GeoDataFrame(
    elderly_cross,
    geometry=gpd.points_from_xy(elderly_cross.lon, elderly_cross.lat),
    crs='EPSG:4326'
)

gdf_cw = gpd.GeoDataFrame(
    cw_node,
    geometry=gpd.points_from_xy(cw_node.lon, cw_node.lat),
    crs='EPSG:4326'
)

# 미터 단위 투영
gdf_acc_m = gdf_acc.to_crs('EPSG:5179')
gdf_cw_m  = gdf_cw.to_crs('EPSG:5179')

# 100m 기준 매칭
T1 = gpd.sjoin_nearest(
    gdf_acc_m,
    gdf_cw_m,
    how='left',
    max_distance=100,        # ← 100m
    distance_col='match_dist'
)

matched = T1['index_right'].notna().sum()
failed  = T1['index_right'].isna().sum()
print(f"매칭 성공: {matched} / {len(T1)}")
print(f"매칭 실패: {failed}")
print(f"평균 매칭 거리: {T1.loc[T1['index_right'].notna(), 'match_dist'].mean():.1f}m")

# 저장
T1_save = T1[T1['index_right'].notna()][['acdnt_no', '노드 ID', 'match_dist']].copy()
T1_save.columns = ['사고ID', '횡단보도ID', '매칭거리']
T1_save.to_csv(T1_PATH, index=False)
print(f"\nT1 저장 완료: {len(T1_save)}건")  # ← 296 나와야 정상

매칭 성공: 296 / 342
매칭 실패: 46
평균 매칭 거리: 31.6m

T1 저장 완료: 296건


## 04. 표준노드링크 중구 + 횡단보도 매칭

In [12]:
# === 노드링크 로드 (서울 중구) ===
link = gpd.read_file(MOCT_LINK_SHP, encoding='cp949')

# 중구 범위로 필터 (좌표 기반)
link = link.to_crs('EPSG:4326')
link_junggu = link.cx[126.96:127.02, 37.54:37.58]
print(f"전국: {len(link)} → 중구: {len(link_junggu)}")
print(link_junggu[['LANES', 'ROAD_RANK', 'MAX_SPD', 'ROAD_NAME']].head())

# 횡단보도 → 가장 가까운 도로 링크 매칭
link_junggu_m = link_junggu.to_crs('EPSG:5179')

cw_road = gpd.sjoin_nearest(
    gdf_cw_m,
    link_junggu_m[['LANES', 'ROAD_RANK', 'MAX_SPD', 'ROAD_NAME', 'geometry']],
    how='left',
    max_distance=30,
    distance_col='road_dist'
)

print(f"\n도로 매칭 성공: {cw_road['LANES'].notna().sum()} / {len(cw_road)}")
print(cw_road[['노드 ID', 'LANES', 'ROAD_RANK', 'MAX_SPD', 'ROAD_NAME']].head(10))

전국: 1554487 → 중구: 3721
      LANES ROAD_RANK  MAX_SPD ROAD_NAME
262       1       104       30      경기대로
919       2       104       50     율곡로2길
955       4       104       50       통일로
1784      3       103       50      새문안로
1785      3       104       50       통일로

도로 매칭 성공: 649 / 674
      노드 ID  LANES ROAD_RANK  MAX_SPD ROAD_NAME
1082  10350    2.0       104     30.0      청계천로
1085  10352    2.0       104     30.0      청계천로
1086  10353    2.0       104     30.0      청계천로
1087  10357    2.0       104     30.0      청계천로
1088  10358    2.0       104     30.0      청계천로
1089  10359    2.0       104     30.0      청계천로
1090   8452    4.0       104     50.0       동호로
1091   8459    2.0       104     50.0       다산로
1092   8469    2.0       104     50.0       다산로
1093   9774    2.0       104     50.0      장충단로


## 05. 고령인구 + 도로등급별 AADT

In [13]:
# === Step E: 고령인구 + 도로등급별 AADT (독립 실행 가능) ===
import sys
sys.path.insert(0, '..')
from utils import *

import pandas as pd
import geopandas as gpd
import os
from shapely.geometry import Point

# ----- 고령인구 (동별) -----
eld = pd.read_csv(ELDERLY_POP_DONG, encoding='utf-8-sig', skiprows=4)
eld.columns = ['시도', '구', '동', '전체_소계', '전체_남', '전체_여',
               '노인_소계', '노인_남', '노인_여',
               '노인_내국_소계', '노인_내국_남', '노인_내국_여',
               '노인_외국_소계', '노인_외국_남', '노인_외국_여']

for col in ['전체_소계', '노인_소계']:
    eld[col] = pd.to_numeric(eld[col], errors='coerce')

# 중구의 동별 데이터만
eld_junggu = eld[(eld['구'] == '중구') & (eld['동'] != '소계')].copy()
eld_junggu['노인비율'] = eld_junggu['노인_소계'] / eld_junggu['전체_소계']

print("=== 중구 동별 고령인구 ===")
print(eld_junggu[['동', '전체_소계', '노인_소계', '노인비율']].to_string(index=False))
print()

# ----- 교통량 12개월 합치기 → AADT -----
hour_cols = [f'{h}시' for h in range(24)]
all_traffic = []
for fname in sorted(os.listdir(TRAFFIC_DIR)):
    if not fname.endswith('.xlsx'):
        continue
    df = pd.read_excel(TRAFFIC_DIR / fname, sheet_name=1)
    all_traffic.append(df)

traffic = pd.concat(all_traffic, ignore_index=True)
traffic['일교통량'] = traffic[hour_cols].apply(
    lambda row: pd.to_numeric(row, errors='coerce').sum(), axis=1
)
aadt = traffic.groupby('지점번호')['일교통량'].mean().reset_index()
aadt.columns = ['지점번호', 'AADT']

# ----- 교통량 지점 ↔ 도로등급 매칭 -----
stations = pd.read_excel(TRAFFIC_DIR / "2025_01.xlsx", sheet_name=2)

link = gpd.read_file(MOCT_LINK_SHP, encoding='cp949')

# ★ 먼저 WGS84로 변환
link = link.to_crs('EPSG:4326')

link_near = link[
    (link.geometry.centroid.x > 126.9) & (link.geometry.centroid.x < 127.1) &
    (link.geometry.centroid.y > 37.5) & (link.geometry.centroid.y < 37.6)
].copy()
print(f"중구 근처 링크: {len(link_near)}")

results = []
for _, st in stations.iterrows():
    pt = Point(st['경도'], st['위도'])
    dists = link_near.geometry.distance(pt)
    if dists.isna().all():
        continue
    nearest_idx = dists.idxmin()
    results.append({
        '지점번호': st['지점번호'],
        'ROAD_RANK': link_near.loc[nearest_idx, 'ROAD_RANK'],
    })

st_road = pd.DataFrame(results)
print(f"매칭 결과: {len(st_road)}개")
print(st_road.head())

st_road = st_road.merge(aadt, on='지점번호', how='left')

grade_aadt = st_road.groupby('ROAD_RANK')['AADT'].mean()
print("\n=== 도로등급별 평균 AADT ===")
print(grade_aadt)

=== 중구 동별 고령인구 ===
   동  전체_소계  노인_소계     노인비율
 소공동   3520    344 0.097727
 회현동   4775   1407 0.294660
  명동   3389    799 0.235763
  필동   5293    973 0.183828
 장충동   5427    967 0.178183
 광희동   7055   1271 0.180156
을지로동   3362    524 0.155860
 신당동   8878   1774 0.199820
 다산동  13481   3223 0.239077
 약수동  15980   4400 0.275344
 청구동  10914   2705 0.247847
신당5동  10284   2131 0.207215
 동화동   9871   2377 0.240806
 황학동  14012   2407 0.171781
 중림동  12203   2321 0.190199



C:\Users\KIMJS\AppData\Local\Temp\ipykernel_3028\1749594918.py:54: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  (link.geometry.centroid.x > 126.9) & (link.geometry.centroid.x < 127.1) &
C:\Users\KIMJS\AppData\Local\Temp\ipykernel_3028\1749594918.py:55: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  (link.geometry.centroid.y > 37.5) & (link.geometry.centroid.y < 37.6)


중구 근처 링크: 24360


C:\Users\KIMJS\AppData\Local\Temp\ipykernel_3028\1749594918.py:62: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  dists = link_near.geometry.distance(pt)


매칭 결과: 147개
   지점번호 ROAD_RANK
0  A-01       103
1  A-02       103
2  A-03       104
3  A-04       104
4  A-05       104

=== 도로등급별 평균 AADT ===
ROAD_RANK
102    39151.087862
103    32310.643014
104    26875.559787
107    67897.638356
Name: AADT, dtype: float64


d:\GitHub\smart-crosswalk\venv\Lib\site-packages\shapely\measurement.py:81: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)


## 06. T2 조립

In [14]:
# === T2 조립 ===
# 횡단보도별 사고 건수 (T1에서 집계)
T1_loaded = pd.read_csv(T1_PATH)
acc_count = T1_loaded.groupby('횡단보도ID').size().reset_index(name='사고건수')

# 기본 테이블: 횡단보도 + 도로특성
T2 = cw_road[['노드 ID', 'lon', 'lat', '읍면동명', 'LANES', 'ROAD_RANK', 'MAX_SPD']].copy()
T2 = T2.rename(columns={'노드 ID': '횡단보도ID'})

# 사고건수 붙이기
T2 = T2.merge(acc_count, on='횡단보도ID', how='left')
T2['사고건수'] = T2['사고건수'].fillna(0).astype(int)

# 도로등급별 AADT 붙이기
T2['추정AADT'] = T2['ROAD_RANK'].map(grade_aadt)

# 노인비율 (중구 전체 단일값)
dong_mapping = {
    '소공동': '소공동', '태평로1가': '소공동', '태평로2가': '소공동', '북창동': '소공동',
    '회현동1가': '회현동', '회현동2가': '회현동', '회현동3가': '회현동', '남창동': '회현동',
    '남대문로1가': '명동', '남대문로2가': '명동', '남대문로3가': '명동',
    '충무로1가': '명동', '충무로2가': '명동',
    '필동1가': '필동', '필동2가': '필동', '필동3가': '필동',
    '남산동2가': '필동', '남산동3가': '필동', '예장동': '필동', '예관동': '필동',
    '충무로3가': '필동', '충무로4가': '필동',
    '장충동1가': '장충동', '장충동2가': '장충동',
    '광희동1가': '광희동', '광희동2가': '광희동',
    '을지로1가': '을지로동', '을지로2가': '을지로동', '을지로3가': '을지로동',
    '을지로4가': '을지로동', '을지로5가': '을지로동', '을지로6가': '을지로동', '을지로7가': '을지로동',
    '수하동': '을지로동', '수표동': '을지로동', '장교동': '을지로동', '삼각동': '을지로동',
    '입정동': '을지로동', '산림동': '을지로동', '무교동': '을지로동', '다동': '을지로동',
    '서소문동': '을지로동', '정동': '을지로동', '순화동': '을지로동', '남학동': '을지로동',
    '주교동': '을지로동', '방산동': '을지로동',
    '신당동': '신당동',
    '쌍림동': '청구동',
    '초동': '동화동', '무학동': '동화동', '인현동1가': '동화동', '인현동2가': '동화동',
    '저동1가': '동화동', '저동2가': '동화동', '주자동': '동화동', '충무로5가': '동화동',
    '황학동': '황학동', '흥인동': '황학동', '오장동': '황학동',
    '중림동': '중림동', '의주로1가': '중림동', '의주로2가': '중림동',
    '만리동1가': '중림동', '만리동2가': '중림동', '충정로1가': '중림동',
    '봉래동1가': '중림동', '봉래동2가': '중림동',
    '남대문로4가': '중림동', '남대문로5가': '중림동',
}

T2['행정동'] = T2['읍면동명'].map(dong_mapping)
unmatched = T2[T2['행정동'].isna()]['읍면동명'].unique()
print(f"매칭 안 된 법정동: {unmatched}")  # ← 이 결과 확인 후 dong_mapping에 추가

dong_ratio = eld_junggu[['동', '노인비율']].rename(columns={'동': '행정동'})
T2 = T2.merge(dong_ratio, on='행정동', how='left')
T2['노인비율'] = T2['노인비율'].fillna(eld_junggu['노인비율'].mean())


print(f"=== T2 완성 ===")
print(f"행: {len(T2)}, 열: {len(T2.columns)}")
print(T2.head(10))
print(f"\n사고 0건 횡단보도: {(T2['사고건수']==0).sum()} / {len(T2)}")
print(f"사고 1건↑ 횡단보도: {(T2['사고건수']>0).sum()}")

T2.to_csv(T2_PATH, index=False)
print(f"T2 저장 완료!")

매칭 안 된 법정동: <StringArray>
[]
Length: 0, dtype: str
=== T2 완성 ===
행: 674, 열: 11
   횡단보도ID         lon        lat   읍면동명  LANES ROAD_RANK  MAX_SPD  사고건수  \
0   10350  126.992528  37.568267    입정동    2.0       104     30.0     0   
1   10352  126.995313  37.568491    산림동    2.0       104     30.0     0   
2   10353  126.996107  37.568559    산림동    2.0       104     30.0     0   
3   10357  127.001861  37.569543    방산동    2.0       104     30.0     1   
4   10358  127.000254  37.569127    주교동    2.0       104     30.0     1   
5   10359  127.002325  37.569435    방산동    2.0       104     30.0     2   
6    8452  127.009042  37.556655    신당동    4.0       104     50.0     0   
7    8459  127.007638  37.549695    신당동    2.0       104     50.0     1   
8    8469  127.009682  37.553173    신당동    2.0       104     50.0     3   
9    9774  127.002234  37.552969  장충동2가    2.0       104     50.0     0   

         추정AADT   행정동      노인비율  
0  26875.559787  을지로동  0.155860  
1  26875.559787  을지로동  0.15